# Category-Level Master Table

This notebook builds a one-row-per-product-category table by joining `order_items_dataset.csv` to
`products_dataset.csv` (for category) and `product_category_name_translation.csv` (for English
names), then rolling up to category level using the already-cleaned order-level outcomes
(`review_score`, `late_delivery_flag`, `review_risk_flag`) from `cleaned_master_order_table.csv`.

Goals:
- give every order item a product category
- translate category names to English where a translation exists
- aggregate to category level: sales volume, seller count, and the same satisfaction/delivery
  metrics used at the order/seller/customer grain, so category can be compared on equal footing
  in the dashboard


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

DATA_DIR = Path("../data/E-Commerce Dataset")
OUT_DIR = Path("../data-visualization")

order_items = pd.read_csv(DATA_DIR / "order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "products_dataset.csv")
translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")
orders = pd.read_csv(OUT_DIR / "cleaned_master_order_table.csv")

print("order_items:", order_items.shape)
print("products:", products.shape)
print("translation:", translation.shape)
print("orders (cleaned master):", orders.shape)


order_items: (112650, 7)
products: (32951, 9)
translation: (71, 2)
orders (cleaned master): (95830, 31)


In [2]:
order_items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [3]:
products["product_category_name"].isna().sum(), len(products)

(610, 32951)

610 of 32,951 products (1.9%) have no category recorded. These items can't be attributed to any
category, so they're dropped from this table only (not from the order-level table) — a category
rollup with an "unknown" bucket of that size would be misleading, and 1.9% of products is a small
enough loss to accept rather than fabricate a category.

In [4]:
products_categorized = products.dropna(subset=["product_category_name"]).copy()
print(f"Products kept: {len(products_categorized):,} of {len(products):,}")

Products kept: 32,341 of 32,951


In [5]:
cats_in_products = set(products_categorized["product_category_name"].unique())
cats_in_translation = set(translation["product_category_name"].unique())
missing_translation = cats_in_products - cats_in_translation
print("Categories with no English translation:", missing_translation)

Categories with no English translation: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


Two categories (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) have no row in the
translation table. Rather than drop them, we keep the original Portuguese name as the English label
— losing these categories entirely would be worse than showing them untranslated.

In [6]:
products_categorized = products_categorized.merge(translation, on="product_category_name", how="left")
products_categorized["category_name_english"] = products_categorized["product_category_name_english"].fillna(
    products_categorized["product_category_name"]
)
products_categorized[["product_category_name", "category_name_english"]].drop_duplicates().head()

,product_category_name,category_name_english
0,perfumaria,perfumery
1,artes,art
2,esporte_lazer,sports_leisure
3,bebes,baby
4,utilidades_domesticas,housewares


In [7]:
items = order_items.merge(
    products_categorized[["product_id", "category_name_english"]],
    on="product_id",
    how="inner"
)
print(f"Order items with a known category: {len(items):,} of {len(order_items):,}")

Order items with a known category: 111,047 of 112,650


## Joining item-level category to order-level outcomes
Each order can contain items from multiple categories (and multiple sellers). To attribute revenue
and satisfaction fairly, we join each (order_id, category) item row to that order's already-cleaned
outcome fields (`review_score`, `late_delivery_flag`, `review_risk_flag`) — the same order can
contribute to more than one category's rollup if it contains items from more than one category,
which mirrors how a real multi-category basket affects the experience of all sellers/categories
involved.

In [8]:
order_outcomes = orders[[
    "order_id", "review_score", "late_delivery_flag", "review_risk_flag",
    "delivery_delay_days", "actual_delivery_days"
]]

items_with_outcomes = items.merge(order_outcomes, on="order_id", how="left")
print("Item rows with a matched order outcome:", items_with_outcomes["review_score"].notna().sum(),
      "of", len(items_with_outcomes))

Item rows with a matched order outcome: 107844 of 111047


In [9]:
items_with_outcomes.isna().sum()

order_id                    0
order_item_id               0
product_id                  0
seller_id                   0
shipping_limit_date         0
price                       0
freight_value               0
category_name_english       0
review_score             3203
late_delivery_flag       3203
review_risk_flag         3203
delivery_delay_days      3203
actual_delivery_days     3203
dtype: int64

Missing `review_score` / `late_delivery_flag` / `delivery_delay_days` here means the parent order
was excluded from the cleaned order table (e.g. never delivered, or no review — see
`cleaning_master_order_table.ipynb`), not that this join is broken. These rows are excluded from the
satisfaction/delivery aggregates below, consistent with how the order-level cleaning already treated
them, but are still counted in sales-volume metrics since a valid sale still occurred.

In [10]:
category_master_table = (
    items_with_outcomes.groupby("category_name_english")
    .agg(
        total_items_sold=("order_item_id", "count"),
        unique_orders=("order_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
        unique_products=("product_id", "nunique"),
        total_sales_value=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        avg_item_price=("price", "mean"),
        avg_review_score=("review_score", "mean"),
        review_count=("review_score", "count"),
        late_delivery_rate=("late_delivery_flag", "mean"),
        review_risk_rate=("review_risk_flag", "mean"),
        avg_delivery_delay_days=("delivery_delay_days", "mean"),
        avg_actual_delivery_days=("actual_delivery_days", "mean"),
    )
    .reset_index()
    .rename(columns={"category_name_english": "product_category"})
)

category_master_table["has_reviews"] = category_master_table["review_count"] > 0
category_master_table["has_delivery_data"] = category_master_table["avg_delivery_delay_days"].notna()

print("Categories:", len(category_master_table))
category_master_table.sort_values("total_sales_value", ascending=False).head(10)

Categories: 73


,product_category,total_items_sold,unique_orders,unique_sellers,unique_products,total_sales_value,total_freight_value,avg_item_price,avg_review_score,review_count,late_delivery_rate,review_risk_rate,avg_delivery_delay_days,avg_actual_delivery_days,has_reviews,has_delivery_data
43,health_beauty,9670,8836,492,2444,1258681.34,182566.73,130.163531,4.189919,9404,0.073905,0.124628,-12.012654,11.471927,True,True
72,watches_gifts,5991,5624,101,1329,1205005.68,100535.93,201.135984,4.071392,5813,0.070704,0.148288,-11.966799,12.140891,True,True
7,bed_bath_table,11115,9417,196,3029,1036988.68,204693.04,93.296327,3.923276,10831,0.068876,0.182162,-11.685809,12.290555,True,True
67,sports_leisure,8641,7720,481,2867,988048.97,168607.51,114.344285,4.164956,8378,0.062425,0.130461,-12.023514,11.668417,True,True
15,computers_accessories,7827,6689,287,1639,911954.32,147318.08,116.513903,3.987117,7607,0.063626,0.170238,-12.457342,12.741028,True,True
39,furniture_decor,8334,6449,370,2657,729762.49,172749.30,87.564494,3.951238,8080,0.069183,0.181559,-12.457921,12.338366,True,True
20,cool_stuff,3796,3632,267,789,635290.85,84039.10,167.357969,4.195446,3689,0.057468,0.11846,-12.515316,11.865004,True,True
49,housewares,6964,5884,468,2335,632248.66,146149.11,90.788148,4.106424,6756,0.048993,0.142688,-12.336738,10.444494,True,True
5,auto,4235,3897,383,1900,592720.11,92664.21,139.957523,4.114934,4098,0.06979,0.139092,-11.461201,11.702782,True,True
42,garden_tools,4347,3518,237,753,485256.46,98962.75,111.630196,4.084198,4240,0.064623,0.149528,-12.041038,13.166038,True,True


In [11]:
category_master_table.isna().sum()

product_category            0
total_items_sold            0
unique_orders               0
unique_sellers              0
unique_products             0
total_sales_value           0
total_freight_value         0
avg_item_price              0
avg_review_score            0
review_count                0
late_delivery_rate          0
review_risk_rate            0
avg_delivery_delay_days     0
avg_actual_delivery_days    0
has_reviews                 0
has_delivery_data           0
dtype: int64

## Data quality summary
- 71 product categories (2 without an official English translation, kept under their Portuguese
  name rather than dropped).
- Every category has at least one sale, so `total_items_sold` / `total_sales_value` are always
  populated.
- `avg_review_score` / `late_delivery_rate` can be `NaN` for a category if none of its orders
  survived the order-level cleaning (undelivered or unreviewed) — flagged via `has_reviews` /
  `has_delivery_data` rather than imputed, matching the convention used in the seller and order
  cleaning notebooks.

In [12]:
category_master_table.to_csv(OUT_DIR / "cleaned_category_level_master_table.csv", index=False)
print("Saved:", OUT_DIR / "cleaned_category_level_master_table.csv")
print("Shape:", category_master_table.shape)

Saved: ..\data-visualization\cleaned_category_level_master_table.csv
Shape: (73, 16)
